# Import and data loading

In [152]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_curve, auc
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, classification_report
from sklearn.model_selection import GridSearchCV
import ast
import warnings
warnings.filterwarnings('ignore')

In [124]:
# Load data
X_train = pd.read_csv('../DATA/tcga/X_train.csv', index_col=0)
X_test = pd.read_csv('../DATA/tcga/X_test.csv', index_col=0)
y_train = pd.read_csv('../DATA/tcga/y_train.csv', index_col=0)["Variant_Type"]
y_test = pd.read_csv('../DATA/tcga/y_test.csv', index_col=0)["Variant_Type"]
y_train_multiclass = pd.read_csv('../DATA/tcga/y_train_initial.csv', index_col=0)
y_test_multiclass = pd.read_csv('../DATA/tcga/y_test_initial.csv', index_col=0)

Data filtering

In [13]:
target_genes = [
    'CDKN1A', 'ABCA12', 'NTPCR', 'PGPEP1', 'RNF19B', 'LCE1E', 'EPN3', 'SCN4B', 'ARVCF', 'FCHO2',
    'PANK2', 'TMEM8B', 'RRM2B', 'ANKRA2', 'ORAI3', 'PLCL2', 'SAC3D1', 'LIMK2', 'FBXO32', 'SCRIB',
    'BHLHE40', 'FCHSD2', 'PAQR7', 'TP53', 'MDM2', 'CCNG1', 'PRKAB1', 'PMAIP1', 'SYTL1', 'LRP1',
    'FHL2', 'SEMA3B', 'BMP7', 'FLRT2', 'PCBP4', 'TP53I11', 'SUSD6', 'CYFIP2', 'PTP4A1', 'PRDM1',
    'TNFRSF10A', 'MCC', 'HES2', 'SLC25A45', 'BORCS7', 'GBE1', 'PERP', 'TRAK1', 'GDF15', 'DRAM1',
    'SESN2', 'RAP2B', 'TNFRSF10D', 'NUPR1', 'KCNN4', 'SLC44A5', 'BTBD10', 'GPC1', 'PLLP', 'TRIP6',
    'BTG2', 'FBXO22', 'SLC30A1', 'RRAD', 'TSPAN11', 'PARD6G', 'KLHDC7A', 'SLC4A11', 'BTG3', 'HES1',
    'POU3F1', 'TSGA10', 'DDB2', 'ISCU', 'SPATA18', 'ZNF219', 'VWCE', 'PHPT1', 'LMNA', 'SLC9A1',
    'C17orf89', 'HRAS', 'PPFIBP1', 'UNC5B', 'GADD45A', 'PHLDA3', 'TGFA', 'ZNF337', 'DDIT4', 'PIDD1',
    'MLF2', 'STAT3', 'CAPN2', 'HSD17B3', 'PPM1J', 'UQCC1', 'PLK3', 'SERPINB5', 'TLR3', 'ACTA2',
    'RAD51C', 'PML', 'MR1', 'STK17A', 'CASP6', 'ICOSLG', 'PPP4R3A', 'VDR', 'TIGAR', 'SERTAD1',
    'TM7SF3', 'EDN2', 'SERPINE1', 'PTPRE', 'MYO6', 'STX6', 'CATSPERG', 'IGFBP7', 'PTAFR', 'YPEL3',
    'RPS27L', 'TRAF4', 'TMEM68', 'ALOX5', 'TNFAIP8', 'PVRL4', 'NEFL', 'TP73', 'CAV1', 'IL1B',
    'RALGDS', 'ZNF195', 'TNFRSF10B', 'TRIM22', 'WDR63', 'ARHGEF3', 'TSKU', 'RETSAT', 'NKAIN4',
    'TRIM32', 'CCNK', 'ISYNA1', 'RBM38', 'ZNF385A', 'TRIAP1', 'CES2', 'ZNF561', 'CERS5', 'PCNA',
    'REV3L', 'PCLO', 'TRIM38', 'CFLAR', 'JAG1', 'RGL1', 'ZNF488', 'ZMAT3', 'CMBL', 'ZNF79',
    'DDR1', 'ACYP2', 'RNASE7', 'PDE4C', 'TRIM5', 'CGB7', 'KRT8', 'RGS20', 'BAX', 'FBXW7',
    'ASCC3', 'DHRS3', 'APAF1', 'SFN', 'PGAP1', 'TYMSOS', 'CHST14', 'KSR1', 'RHOC', 'PGF',
    'HSPA4L', 'ACER2', 'DUSP14', 'APOBEC3H', 'TNFRSF10C', 'PLCXD2', 'AKAP9', 'COBLL1', 'LACC1',
    'RPS19', 'POLH', 'KITLG', 'ANXA4', 'E2F7', 'BCL2L1', 'TRIML2', 'PLEKHF1', 'CCDC51', 'CPEB2',
    'LPXN', 'SARS', 'PPM1D', 'SLC12A4', 'APOBEC3C', 'EPS8L2', 'BCL6', 'VCAN', 'PLTP', 'CDH8',
    'CPSF4', 'LRPAP1', 'SCIN', 'SULF2', 'ATF3', 'ASTN2', 'FAM210B', 'BLCAP', 'ADCK3', 'PLXNB1',
    'DUSP11', 'DNAJB2', 'MFAP3L', 'SCN3B', 'XPC', 'BBC3', 'CD82', 'GLS2', 'C17orf82', 'AK3',
    'PLXNB2', 'GCC2', 'DOCK8', 'MKNK2', 'SDC4', 'AEN', 'CCDC90B', 'CDIP1', 'GPX1', 'COL7A1',
    'ALDH1A3', 'PRKAB2', 'METTL8', 'DUSP5', 'MON2', 'SDPR', 'BLOC1S2', 'DYRK3', 'CPE', 'GRHL3',
    'CPEB4', 'BBS2', 'PRKX', 'PPP1R3C', 'DUSP7', 'MRPL49', 'SMAD3', 'FAS', 'EDA2R', 'CSF1',
    'HHAT', 'CSNK1G1', 'BTG1', 'PRODH', 'STEAP3', 'EBI3', 'MYBPHL', 'SNX2', 'GPR87', 'EPHA2',
    'DCP1B', 'IGDCC4', 'DGKA', 'CEL', 'PTPRU', 'ABHD4', 'EFNB1', 'MYLK', 'SOCS4', 'NINJ1',
    'FAM13C', 'ENC1', 'IKBIP', 'FAM49A', 'CLCA2', 'RGMA', 'ABTB2', 'EI24', 'MYOF', 'TAB3',
    'PLK2', 'FAM198B', 'FOSL1', 'LAPTM5', 'FAM84B', 'CLDN1', 'RGS16', 'ADGRG1', 'EML2', 'NFKBIA',
    'TCAIM', 'PSTPIP2', 'FAM212B', 'FUCA1', 'MAST4', 'GNAI1', 'CLP1', 'RND3', 'AIFM2', 'ENPP2',
    'NHLH2', 'TEP1', 'SESN1', 'FDXR', 'IER5', 'MICALL1', 'INPP1', 'CROT', 'RNF144B', 'AMOTL1',
    'ETV7', 'NLRP1', 'TET2', 'TP53I3', 'LIF', 'PADI4', 'NOTCH1', 'ITGA3', 'CYP4F3', 'S100A2',
    'AMZ2', 'FAM196A', 'NYNRIN', 'TEX9', 'TP53INP1', 'NADSYN1', 'PANK1', 'RABGGTA', 'KRT15',
    'DAPK1', 'SCN2A', 'ARC', 'FAM98C', 'P3H2', 'TMEM63B'
]

In [41]:
# Filter dataset to only include genes that exist in X_train
available_genes = [gene for gene in target_genes if gene in X_train.columns]
X_train_target = X_train[available_genes]
X_test_target = X_test[available_genes]

# Task 1 - binary classification

## XGBoost

In [ ]:
xgb_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'reg_lambda': [1, 5],
    'reg_alpha': [0, 1]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train_target, y_train)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_target)
y_proba = best_model.predict_proba(X_test_target)[:, 1]

print("Best Hyperparameters:", grid_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("Classification Report:\n", classification_report(y_test, y_pred))

Best Hyperparameters: {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 200, 'reg_alpha': 0, 'reg_lambda': 1, 'subsample': 0.8}
Accuracy: 0.9045643153526971
ROC AUC: 0.9470496894409938
Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.94      0.93       161
           1       0.88      0.82      0.85        80

    accuracy                           0.90       241
   macro avg       0.90      0.88      0.89       241
weighted avg       0.90      0.90      0.90       241



### Random forest

In [44]:
rf_model = RandomForestClassifier(random_state=42)

# Define hyperparameter grid for tuning
param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 5],
    'max_features': ['sqrt', 0.5, 1.0],
    'bootstrap': [True]
}

# Set up cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid search with cross-validation
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    verbose=1,
    n_jobs=-1
)

# Fit the model
grid_search.fit(X_train_target, y_train)

# Evaluate the best model
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test_target)
y_proba = best_rf.predict_proba(X_test_target)[:, 1]

print("Best Hyperparameters:", grid_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("Classification Report:\n", classification_report(y_test, y_pred))

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best Hyperparameters: {'bootstrap': True, 'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 300}
Accuracy: 0.8589211618257261
ROC AUC: 0.9385093167701863
Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.92      0.90       161
           1       0.82      0.74      0.78        80

    accuracy                           0.86       241
   macro avg       0.85      0.83      0.84       241
weighted avg       0.86      0.86      0.86       241



## MLP

In [ ]:
mlp = MLPClassifier(
    random_state=42,
    early_stopping=True,       # Automatically set aside a validation split
    validation_fraction=0.1,   # 10% of train → internal validation
    max_iter=200,              # Stop after 200 epochs if no improvement
    tol=1e-4,                  # Early‐stopping tolerance
    verbose=False
)

param_grid = {
    'hidden_layer_sizes': [(256, 128), (512, 256), (128,)],
    'activation': ['relu', 'tanh'],
    'alpha': [1e-4, 1e-3, 1e-2],
    'learning_rate_init': [1e-3, 5e-4],
    'learning_rate': ['constant', 'adaptive'],
    'batch_size': [32, 64]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search_nn = GridSearchCV(
    estimator=mlp,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1
)


grid_search_nn.fit(X_train_target, y_train)

best_mlp = grid_search_nn.best_estimator_

y_pred  = best_mlp.predict(X_test_target)
y_proba = best_mlp.predict_proba(X_test_target)[:, 1]

print("Best Hyperparameters (MLP):", grid_search_nn.best_params_)
print("MLP Accuracy:", accuracy_score(y_test, y_pred))
print("MLP ROC AUC:", roc_auc_score(y_test, y_proba))
print("MLP Classification Report:\n", classification_report(y_test, y_pred))

Best Hyperparameters (MLP): {'activation': 'tanh', 'alpha': 0.001, 'batch_size': 32, 'hidden_layer_sizes': (256, 128), 'learning_rate': 'constant', 'learning_rate_init': 0.001}
MLP Accuracy: 0.8381742738589212
MLP ROC AUC: 0.8784937888198758
MLP Classification Report:
               precision    recall  f1-score   support

           0       0.86      0.91      0.88       161
           1       0.79      0.70      0.74        80

    accuracy                           0.84       241
   macro avg       0.82      0.80      0.81       241
weighted avg       0.84      0.84      0.84       241



## Andrea's embeddings 

In [ ]:
gene_emb_df = pd.read_csv("../DATA/embeddings_dataset.csv", sep=';')
gene_emb_df

In [79]:
shared_genes = [g for g in X_train_target.columns if g in gene_emb_df["gene_name"].values]
missing_genes = [g for g in X_train_target.columns if g not in gene_emb_df["gene_name"].values]
print(f"Using {len(shared_genes)}/{len(X_train_target.columns)} gene embeddings.")

Using 301/335 gene embeddings.


In [82]:
filtered_embeddings_df = gene_emb_df.set_index("gene_name").loc[shared_genes]
filtered_embeddings_df = filtered_embeddings_df.reindex(shared_genes)

In [ ]:
# Convert string representations back to arrays
embeddings_list = [ast.literal_eval(emb) for emb in filtered_embeddings_df["embeddings"].values]
embeddings_matrix = np.vstack(embeddings_list)
print("Embeddings matrix shape:", embeddings_matrix.shape)

Embeddings matrix shape: (301, 768)


In [92]:
def compute_cell_embeddings_from_df(expr_df, embeddings_matrix, shared_genes):
    """
    expr_df: pandas DataFrame (n_cells, n_genes_subset), columns include shared_genes
    embeddings_matrix: np.array of shape (n_shared_genes, embed_dim)
    shared_genes: list of gene names in exactly the same order as embeddings_matrix rows

    Returns:
    - cell_embeddings: np.array of shape (n_cells, embed_dim)
    """
    # 4a. Extract only the shared_genes columns and convert to numpy array
    expr_shared = expr_df[shared_genes].values  # shape = (n_cells, n_shared_genes)

    # 4b. Normalize counts by total counts per cell (avoid zero‐division by adding 1e-6)
    row_sums = expr_shared.sum(axis=1, keepdims=True)  # shape = (n_cells, 1)
    normalized_counts = expr_shared / (row_sums + 1e-6)  # shape = (n_cells, n_shared_genes)

    # 4c. Compute weighted sum: (n_cells × n_genes) dot (n_genes × embed_dim)
    cell_embeddings = normalized_counts.dot(embeddings_matrix)  # shape = (n_cells, embed_dim)
    return cell_embeddings

In [93]:
X_train_emb = compute_cell_embeddings_from_df(X_train_target, embeddings_matrix, shared_genes)
X_test_emb  = compute_cell_embeddings_from_df(X_test_target, embeddings_matrix, shared_genes)

print("X_train_emb shape:", X_train_emb.shape)  # (n_train_cells, embedding_dim)
print("X_test_emb shape:",  X_test_emb.shape) 

X_train_emb shape: (962, 768)
X_test_emb shape: (241, 768)


In [ ]:
mlp = MLPClassifier(
    random_state=42,
    early_stopping=True,       # Automatically set aside a validation split
    validation_fraction=0.1,   # 10% of train → internal validation
    max_iter=200,              # Stop after 200 epochs if no improvement
    tol=1e-4,                  # Early‐stopping tolerance
    verbose=False
)

param_grid = {
    'hidden_layer_sizes': [(256, 128), (512, 256), (128,)],
    'activation': ['relu', 'tanh'],
    'alpha': [1e-4, 1e-3, 1e-2],
    'learning_rate_init': [1e-3, 5e-4],
    'learning_rate': ['constant', 'adaptive'],
    'batch_size': [32, 64]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search_nn = GridSearchCV(
    estimator=mlp,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1
)


grid_search_nn.fit(X_train_emb, y_train)

best_mlp_emb = grid_search_nn.best_estimator_

y_pred  = best_mlp_emb.predict(X_test_emb)
y_proba = best_mlp_emb.predict_proba(X_test_emb)[:, 1]

print("Best Hyperparameters (MLP):", grid_search_nn.best_params_)
print("MLP Accuracy:", accuracy_score(y_test, y_pred))
print("MLP ROC AUC:", roc_auc_score(y_test, y_proba))
print("MLP Classification Report:\n", classification_report(y_test, y_pred))

Best Hyperparameters (MLP): {'activation': 'tanh', 'alpha': 0.001, 'batch_size': 64, 'hidden_layer_sizes': (512, 256), 'learning_rate': 'constant', 'learning_rate_init': 0.001}
MLP Accuracy: 0.8298755186721992
MLP ROC AUC: 0.8944099378881987
MLP Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.86      0.87       161
           1       0.73      0.78      0.75        80

    accuracy                           0.83       241
   macro avg       0.81      0.82      0.81       241
weighted avg       0.83      0.83      0.83       241



## Ensemble

###  Voting classifier

In [ ]:
rf_clf  = best_rf   
xgb_clf = best_model  
mlp_clf = best_mlp    

# Package them into a VotingClassifier
voting_clf = VotingClassifier(
    estimators=[
        ('rf',  rf_clf),
        ('xgb', xgb_clf),
        ('mlp', mlp_clf)
    ],
    voting='soft',   # 'soft' = average predicted probabilities; 'hard' = majority class vote
    weights=[2, 3, 1],  # you can adjust if e.g. xgb is more accurate than rf
    n_jobs=-1
)

voting_clf.fit(X_train_target, y_train)  

y_pred_ens = voting_clf.predict(X_test_target)
y_proba_ens = voting_clf.predict_proba(X_test_target)[:, 1]

print("Ensemble Accuracy:", accuracy_score(y_test, y_pred_ens))
print("Ensemble ROC AUC:", roc_auc_score(y_test, y_proba_ens))
print("Ensemble Classification Report:\n", classification_report(y_test, y_pred_ens))

✅ Ensemble Accuracy: 0.8589211618257261
📊 Ensemble ROC AUC: 0.9387422360248447
📋 Ensemble Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.93      0.90       161
           1       0.84      0.71      0.77        80

    accuracy                           0.86       241
   macro avg       0.85      0.82      0.83       241
weighted avg       0.86      0.86      0.86       241



### Fusion models

In [159]:
meta_mlp_no_hidden = MLPClassifier(
    hidden_layer_sizes=(),    # no hidden layers
    activation='relu',        # activation is ignored because no hidden units exist
    solver='lbfgs',           # use lbfgs for small “linear” network
    alpha=1e-3,               # L2 penalty (similar to C in logistic regression)
    random_state=42
)

In [164]:
meta_mlp_one_hidden = MLPClassifier(
    hidden_layer_sizes=(16,),  # one hidden layer with 16 units
    activation='relu',
    alpha=1e-3,
    learning_rate_init=1e-3,
    max_iter=200,
    early_stopping=True,
    random_state=42
)

In [165]:
stack_clf = StackingClassifier(
    estimators=[
        ('rf',  best_rf),
        ('xgb', best_model),
        ('mlp', best_mlp)
    ],
    final_estimator=meta_mlp_one_hidden,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    passthrough=False,   # meta‐model only sees base models’ predictions
    n_jobs=-1,
    verbose=1
)

# Fit on your train set: (X_train_target, y_train)
stack_clf.fit(X_train_target, y_train)

# Evaluate on held‐out test set:
y_pred_stack  = stack_clf.predict(X_test_target)
y_proba_stack = stack_clf.predict_proba(X_test_target)[:, 1]

print("Stacking Ensemble Accuracy:", accuracy_score(y_test, y_pred_stack))
print("Stacking Classification Report:\n", classification_report(y_test, y_pred_stack))

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    3.4s finished
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    2.9s finished
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   12.5s finished


Stacking Ensemble Accuracy: 0.8
Stacking Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         9
           1       0.00      0.00      0.00         4
           2       0.72      0.59      0.65        66
           3       0.84      0.95      0.89       161

    accuracy                           0.80       240
   macro avg       0.39      0.39      0.38       240
weighted avg       0.76      0.80      0.78       240



# Task 2 - multiclass classification

In [133]:
y_train = y_train_multiclass
y_test = y_test_multiclass

In [134]:
print(f"Training data shape: {X_train_target.shape}")
print(f"Training labels distribution:\n{y_train.value_counts()}")
print()
print(f"Testing data shape: {X_test_target.shape}")
print(f"Testing labels distribution:\n{y_test.value_counts()}")

Training data shape: (962, 335)
Training labels distribution:
Variant_Type
WT              642
SNP             265
DEL              37
INS              14
DNP               4
Name: count, dtype: int64

Testing data shape: (241, 335)
Testing labels distribution:
Variant_Type
WT              161
SNP              66
DEL               9
INS               4
DNP               1
Name: count, dtype: int64


The number of samples with the 'DNP' mutation is very small, so we remove them to prevent their disproportionate influence on the classification model, which could negatively impact performance.

In [135]:
train_mask = y_train.iloc[:, 0] != 'DNP'
X_train_target = X_train_target.loc[train_mask].reset_index(drop=True)
y_train = y_train.loc[train_mask].reset_index(drop=True)

test_mask = y_test.iloc[:, 0] != 'DNP'
X_test_target = X_test_target.loc[test_mask].reset_index(drop=True)
y_test = y_test.loc[test_mask].reset_index(drop=True)

In [136]:
# Flatten and encode
y_train_flat = y_train.iloc[:, 0].values
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_flat)

y_test_flat = y_test.iloc[:, 0].values
y_test = label_encoder.transform(y_test_flat)

### XGBoost

In [138]:
num_classes = 4

xgb_model = XGBClassifier(
    objective='multi:softprob',
    num_class=num_classes,
    eval_metric='mlogloss',
    random_state=42
)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'reg_lambda': [1, 5],
    'reg_alpha': [0, 1]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='f1_weighted',
    cv=cv,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train_target, y_train)

xgb_model_multiclass = grid_search.best_estimator_
y_pred = xgb_model_multiclass.predict(X_test_target)
y_proba = xgb_model_multiclass.predict_proba(X_test_target)

print("Best Parameters:", grid_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Fitting 5 folds for each of 288 candidates, totalling 1440 fits
Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 200, 'reg_alpha': 0, 'reg_lambda': 5, 'subsample': 0.8}
Accuracy: 0.8041666666666667
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         9
           1       0.00      0.00      0.00         4
           2       0.68      0.67      0.67        66
           3       0.86      0.93      0.89       161

    accuracy                           0.80       240
   macro avg       0.38      0.40      0.39       240
weighted avg       0.76      0.80      0.78       240



### Random forest

In [140]:
rf_model = RandomForestClassifier(random_state=42)

# Define hyperparameter grid for tuning
param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 5],
    'max_features': ['sqrt', 0.5, 1.0],
    'bootstrap': [True]
}

# Set up cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid search with cross-validation
grid_search = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid,
    scoring='f1_weighted',
    cv=cv,
    verbose=1,
    n_jobs=-1
)

# Fit the model
grid_search.fit(X_train_target, y_train)

# Evaluate the best model
rf_model_multiclass = grid_search.best_estimator_
y_pred = rf_model_multiclass.predict(X_test_target)
y_proba = rf_model_multiclass.predict_proba(X_test_target)[:, 1]

print("Best Hyperparameters:", grid_search.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best Hyperparameters: {'bootstrap': True, 'max_depth': 10, 'max_features': 0.5, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
Accuracy: 0.8333333333333334
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         9
           1       0.00      0.00      0.00         4
           2       0.72      0.73      0.72        66
           3       0.88      0.94      0.91       161

    accuracy                           0.83       240
   macro avg       0.40      0.42      0.41       240
weighted avg       0.79      0.83      0.81       240



### MLP

In [141]:
mlp = MLPClassifier(
    random_state=42,
    early_stopping=True,       # Automatically set aside a validation split
    validation_fraction=0.1,   # 10% of train → internal validation
    max_iter=200,              # Stop after 200 epochs if no improvement
    tol=1e-4,                  # Early‐stopping tolerance
    verbose=False
)

param_grid = {
    'hidden_layer_sizes': [(256, 128), (512, 256), (128,)],
    'activation': ['relu', 'tanh'],
    'alpha': [1e-4, 1e-3, 1e-2],
    'learning_rate_init': [1e-3, 5e-4],
    'learning_rate': ['constant', 'adaptive'],
    'batch_size': [32, 64]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search_nn = GridSearchCV(
    estimator=mlp,
    param_grid=param_grid,
    scoring='f1_weighted',
    cv=cv,
    n_jobs=-1
)


grid_search_nn.fit(X_train_target, y_train)

mlp_model_multiclass = grid_search_nn.best_estimator_

y_pred  = mlp_model_multiclass.predict(X_test_target)
y_proba = mlp_model_multiclass.predict_proba(X_test_target)[:, 1]

print("Best Hyperparameters:", grid_search_nn.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Best Hyperparameters: {'activation': 'tanh', 'alpha': 0.001, 'batch_size': 32, 'hidden_layer_sizes': (128,), 'learning_rate': 'constant', 'learning_rate_init': 0.001}
Accuracy: 0.7708333333333334
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         9
           1       0.00      0.00      0.00         4
           2       0.64      0.56      0.60        66
           3       0.81      0.92      0.86       161

    accuracy                           0.77       240
   macro avg       0.36      0.37      0.36       240
weighted avg       0.72      0.77      0.74       240



### Andrea's embeddings 

In [144]:
gene_emb_df = pd.read_csv("../DATA/embeddings_dataset.csv", sep=';')
gene_emb_df

,ensembl_id,gene_name,description,embeddings
0,ENSG00000273106,PPM1B-DT,"is an rna gene, and is affiliated with the lnc...","[0.7153338, -0.11398319, -2.0761514, 0.5069085..."
1,ENSG00000038532,CLEC16A,this gene encodes member of the domain family....,"[0.36557245, 1.0013258, -0.0023882652, 0.54005..."
2,ENSG00000181856,SLC2A4,this gene is member of the carrier family faci...,"[-0.21235171, 2.9689298, 0.035055883, -0.12451..."
3,ENSG00000262703,ENSG00000262703,"novel transcript, antisense to rmi2 is an rna ...","[0.3950196, 1.5300467, -1.4824657, 1.1461742, ..."
4,ENSG00000090661,CERS4,enables sphingosine n-acyltransferase activity...,"[1.4770727, 0.21068904, -0.44252306, -0.483786..."
...,...,...,...,...
22835,ENSG00000275111,ZNF2,the protein encoded by this gene belongs to th...,"[-0.37485185, 0.9992399, 0.3704633, 0.41376647..."
22836,ENSG00000171953,ATPAF2,this gene encodes factor for the f1 component ...,"[-0.44170868, -0.9684407, 0.9719296, -0.525657..."
22837,ENSG00000198650,TAT,nuclear gene encodes mitochondrial protein whi...,"[0.36593568, 1.2355629, -0.5669457, 0.7016244,..."
22838,ENSG00000226696,LENG8-AS1,"rna is rna gene, is with the class.","[1.0164974, 0.8513611, -1.4877115, -1.7738836,..."


In [145]:
shared_genes = [g for g in X_train_target.columns if g in gene_emb_df["gene_name"].values]
missing_genes = [g for g in X_train_target.columns if g not in gene_emb_df["gene_name"].values]
print(f"Using {len(shared_genes)}/{len(X_train_target.columns)} gene embeddings.")

Using 301/335 gene embeddings.


In [146]:
filtered_embeddings_df = gene_emb_df.set_index("gene_name").loc[shared_genes]
filtered_embeddings_df = filtered_embeddings_df.reindex(shared_genes)

In [147]:
# Convert string representations back to arrays
embeddings_list = [ast.literal_eval(emb) for emb in filtered_embeddings_df["embeddings"].values]
embeddings_matrix = np.vstack(embeddings_list)
print("Embeddings matrix shape:", embeddings_matrix.shape)

Embeddings matrix shape: (301, 768)


In [148]:
def compute_cell_embeddings_from_df(expr_df, embeddings_matrix, shared_genes):
    """
    expr_df: pandas DataFrame (n_cells, n_genes_subset), columns include shared_genes
    embeddings_matrix: np.array of shape (n_shared_genes, embed_dim)
    shared_genes: list of gene names in exactly the same order as embeddings_matrix rows

    Returns:
    - cell_embeddings: np.array of shape (n_cells, embed_dim)
    """
    # 4a. Extract only the shared_genes columns and convert to numpy array
    expr_shared = expr_df[shared_genes].values  # shape = (n_cells, n_shared_genes)

    # 4b. Normalize counts by total counts per cell (avoid zero‐division by adding 1e-6)
    row_sums = expr_shared.sum(axis=1, keepdims=True)  # shape = (n_cells, 1)
    normalized_counts = expr_shared / (row_sums + 1e-6)  # shape = (n_cells, n_shared_genes)

    # 4c. Compute weighted sum: (n_cells × n_genes) dot (n_genes × embed_dim)
    cell_embeddings = normalized_counts.dot(embeddings_matrix)  # shape = (n_cells, embed_dim)
    return cell_embeddings

In [149]:
X_train_emb = compute_cell_embeddings_from_df(X_train_target, embeddings_matrix, shared_genes)
X_test_emb  = compute_cell_embeddings_from_df(X_test_target, embeddings_matrix, shared_genes)

print("X_train_emb shape:", X_train_emb.shape)  # (n_train_cells, embedding_dim)
print("X_test_emb shape:",  X_test_emb.shape) 

X_train_emb shape: (958, 768)
X_test_emb shape: (240, 768)


In [150]:
mlp = MLPClassifier(
    random_state=42,
    early_stopping=True,       # Automatically set aside a validation split
    validation_fraction=0.1,   # 10% of train → internal validation
    max_iter=200,              # Stop after 200 epochs if no improvement
    tol=1e-4,                  # Early‐stopping tolerance
    verbose=False
)

param_grid = {
    'hidden_layer_sizes': [(256, 128), (512, 256), (128,)],
    'activation': ['relu', 'tanh'],
    'alpha': [1e-4, 1e-3, 1e-2],
    'learning_rate_init': [1e-3, 5e-4],
    'learning_rate': ['constant', 'adaptive'],
    'batch_size': [32, 64]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search_nn = GridSearchCV(
    estimator=mlp,
    param_grid=param_grid,
    scoring='f1_weighted',
    cv=cv,
    n_jobs=-1
)


grid_search_nn.fit(X_train_emb, y_train)

mlp_emb_model_multiclass = grid_search_nn.best_estimator_

y_pred  = mlp_emb_model_multiclass.predict(X_test_emb)
y_proba = mlp_emb_model_multiclass.predict_proba(X_test_emb)[:, 1]

print("Best Hyperparameters:", grid_search_nn.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Best Hyperparameters: {'activation': 'relu', 'alpha': 0.001, 'batch_size': 32, 'hidden_layer_sizes': (512, 256), 'learning_rate': 'constant', 'learning_rate_init': 0.001}
Accuracy: 0.7791666666666667
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         9
           1       0.00      0.00      0.00         4
           2       0.61      0.67      0.64        66
           3       0.85      0.89      0.87       161

    accuracy                           0.78       240
   macro avg       0.37      0.39      0.38       240
weighted avg       0.74      0.78      0.76       240



## Ensemble

In [155]:
meta_mlp_no_hidden = MLPClassifier(
    hidden_layer_sizes=(),    # no hidden layers
    activation='relu',        # activation is ignored because no hidden units exist
    solver='lbfgs',           # use lbfgs for small “linear” network
    alpha=1e-3,               # L2 penalty (similar to C in logistic regression)
    random_state=42
)

In [162]:
meta_mlp_one_hidden = MLPClassifier(
    hidden_layer_sizes=(16,),  # one hidden layer with 16 units
    activation='relu',
    alpha=1e-3,
    learning_rate_init=1e-3,
    max_iter=200,
    early_stopping=True,
    random_state=42
)

In [157]:
stack_clf = StackingClassifier(
    estimators=[
        ('rf',  rf_model_multiclass),
        ('xgb', xgb_model_multiclass),
        ('mlp', mlp_model_multiclass)
    ],
    final_estimator=meta_mlp_no_hidden,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    passthrough=False,   # meta‐model only sees base models’ predictions
    n_jobs=-1,
    verbose=1
)

# Fit on your train set: (X_train_target, y_train)
stack_clf.fit(X_train_target, y_train)

# Evaluate on held‐out test set:
y_pred_stack  = stack_clf.predict(X_test_target)
y_proba_stack = stack_clf.predict_proba(X_test_target)[:, 1]

print("Stacking Ensemble Accuracy:", accuracy_score(y_test, y_pred_stack))
print("Stacking Classification Report:\n", classification_report(y_test, y_pred_stack))

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    2.3s finished
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:    4.6s finished
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   12.5s finished


Stacking Ensemble Accuracy: 0.8125
Stacking Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         9
           1       0.00      0.00      0.00         4
           2       0.68      0.70      0.69        66
           3       0.87      0.93      0.89       161

    accuracy                           0.81       240
   macro avg       0.39      0.41      0.40       240
weighted avg       0.77      0.81      0.79       240



In [158]:
if isinstance(stack_clf.final_estimator_, MLPClassifier) and stack_clf.final_estimator_.hidden_layer_sizes == ():
    coefs = stack_clf.final_estimator_.coefs_[0].flatten()
    bias  = stack_clf.final_estimator_.intercepts_[0][0]
    print("Meta‐MLP Weights (just a linear combination of base‐probabilities):")
    print(f"  RF weight = {coefs[0]:.3f}")
    print(f"  XGB weight = {coefs[1]:.3f}")
    print(f"  MLP weight = {coefs[2]:.3f}")
    print(f"  Bias term = {bias:.3f}")
    print("Interpretation: final_logit = RF_w * p_rf + XGB_w * p_xgb + MLP_w * p_mlp + bias")


Meta‐MLP Weights (just a linear combination of base‐probabilities):
  RF weight = 10.189
  XGB weight = -1.204
  MLP weight = -1.262
  Bias term = -1.068
Interpretation: final_logit = RF_w * p_rf + XGB_w * p_xgb + MLP_w * p_mlp + bias
